# 🤖 Sistema Multi-Agente para un Portfolio Personal

**Demo técnica**: orquestador + subagentes especializados (RAG, gestión de contacto,
análisis de datos), con memoria de conversación, *tool calling* y *guardrails*.

Este notebook está pensado como **pieza de portfolio**: se ejecuta de principio a fin
sin necesidad de ninguna API key (usa un modo *offline* de respaldo), pero está
preparado para conectarse a un LLM real (GPT-4o-mini, Groq/Llama 3, etc.) en cuanto
definas la variable de entorno correspondiente.

**Stack demostrado:** arquitectura multi-agente · RAG (chunking + embeddings +
similitud coseno) · memoria de ventana · *tool calling* · *guardrails* ·
cumplimiento RGPD · despliegue (Hugging Face Spaces / Streamlit / Docker).

---

## Índice

1. [Arquitectura del sistema](#1)
2. [Configuración de modelos (LLM)](#2)
3. [Base de conocimiento y RAG](#3)
4. [Memoria de conversación](#4)
5. [Herramientas (Tool Calling)](#5)
6. [Subagentes especializados](#6)
7. [Agente orquestador](#7)
8. [Demo: conversación de ejemplo](#8)
9. [Analista de datos: ejemplo con CSV](#9)
10. [Guardrails y cumplimiento (RGPD)](#10)
11. [Despliegue gratuito](#11)
12. [Integración en la web (embed)](#12)
13. [Conclusión](#13)


<a id="1"></a>
## 1. Arquitectura del sistema

Se usa un patrón de **orquestación multi-agente**: un agente principal entiende la
intención del visitante y delega en el subagente especializado adecuado.

```
                        ┌───────────────────────────┐
                        │   Visitante de la web      │
                        └────────────┬───────────────┘
                                     │  mensaje
                                     ▼
                        ┌───────────────────────────┐
                        │   AGENTE ORQUESTADOR        │
                        │  · entiende la intención    │
                        │  · mantiene memoria         │
                        │  · aplica guardrails         │
                        └───┬───────────┬───────────┬─┘
                            │           │           │
                 ┌──────────▼──┐ ┌──────▼──────┐ ┌──▼───────────────┐
                 │ RAG Experto  │ │  Contacto   │ │ Analista de Datos │
                 │ (trayectoria)│ │ (Airtable/  │ │ (CSV / Excel)      │
                 │              │ │  Sheets)    │ │                    │
                 └──────────────┘ └─────────────┘ └────────────────────┘
```

- **Orquestador**: clasifica la intención (RAG / contacto / datos / desconocida) y
  enruta la conversación, exponiendo *qué* subagente se invoca en cada turno
  (transparencia de proceso).
- **Subagente RAG**: responde preguntas sobre experiencia y proyectos consultando
  una base de conocimiento vectorizada (CV + descripciones de proyectos).
- **Subagente de Contacto**: captura nombre / email / presupuesto / interés y los
  envía a una herramienta externa (Airtable o Google Sheets).
- **Subagente Analista de Datos**: procesa un CSV/Excel subido por el visitante y
  devuelve un mini-informe automático.


<a id="2"></a>
## 2. Configuración del entorno y modelos

Para minimizar costes se recomienda un modelo eficiente como **GPT-4o-mini** (OpenAI)
o un modelo gratuito servido por **Groq** (p. ej. `llama-3.1-8b-instant`).

Este notebook no depende de librerías pesadas (no usa `langchain`, `numpy` ni
`sklearn`) para que puedas leerlo y ejecutarlo en cualquier entorno — Colab,
Hugging Face Spaces, tu portátil— sin fricción. La lógica de *chunking*,
*embeddings* y similitud está implementada en Python puro para que se vea
exactamente qué ocurre "bajo el capó"; en producción se sustituiría por
`text-embedding-3-small` + una base vectorial como **Supabase (pgvector)**.

```bash
# Dependencias opcionales para producción:
# pip install openai groq pandas supabase
```


In [1]:
import os
import re
import csv
import json
import math
import textwrap
from collections import deque, Counter
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Callable

# Directorio de trabajo para los artefactos que genera la demo (contactos, logs...)
DEMO_DIR = Path("demo_artifacts")
DEMO_DIR.mkdir(exist_ok=True)

# Modelo "por defecto" a usar si hay una API key disponible.
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")

print(f"Modelo configurado: {LLM_MODEL}")
print(f"OPENAI_API_KEY presente: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"GROQ_API_KEY presente:   {bool(os.getenv('GROQ_API_KEY'))}")


Modelo configurado: gpt-4o-mini
OPENAI_API_KEY presente: False
GROQ_API_KEY presente:   False


In [2]:
def call_llm(system_prompt: str, user_prompt: str, model: str = LLM_MODEL) -> str:
    """
    Wrapper único para llamar a un LLM.

    - Si existe OPENAI_API_KEY -> usa la API de OpenAI (GPT-4o-mini).
    - Si existe GROQ_API_KEY  -> usa la API de Groq (Llama 3, gratuita).
    - Si no hay ninguna key    -> modo "offline" determinista, para que la demo
      completa se pueda ejecutar y evaluar sin gastar ni un céntimo ni requerir
      credenciales (útil para revisores del portfolio).

    Este único punto de entrada es lo que le permite a cualquier subagente ser
    agnóstico del proveedor de LLM.
    """
    if os.getenv("OPENAI_API_KEY"):
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.3,
        )
        return resp.choices[0].message.content

    if os.getenv("GROQ_API_KEY"):
        from groq import Groq
        client = Groq()
        resp = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.3,
        )
        return resp.choices[0].message.content

    # --- Modo offline (fallback determinista, sin coste ni API key) ---
    return _offline_llm_stub(system_prompt, user_prompt)


def _offline_llm_stub(system_prompt: str, user_prompt: str) -> str:
    """Respuesta plantilla para poder ejecutar la demo sin conexión a un LLM real."""
    return (
        "[modo offline] He recibido tu consulta y la información recuperada por el "
        "subagente correspondiente. Con una API key real (OPENAI_API_KEY o "
        "GROQ_API_KEY) esta respuesta la redactaría el LLM en lenguaje natural."
    )


<a id="3"></a>
## 3. Base de conocimiento y RAG (Retrieval-Augmented Generation)

Simulamos el "Data Loader" cargando el CV y las descripciones de proyectos como
texto plano (en producción serían PDFs procesados con un loader tipo
`PyPDFLoader`). Después:

1. **Text splitting**: se trocea el texto en fragmentos pequeños con solapamiento.
2. **Embeddings**: cada fragmento se convierte en un vector numérico. Aquí usamos
   un vectorizador *bag-of-words* implementado a mano (para que el mecanismo sea
   transparente); en producción sería `text-embedding-3-small` de OpenAI.
3. **Vector store**: los vectores se guardan e indexan (aquí en memoria; en
   producción, en **Supabase** con `pgvector`, que tiene tier gratuito).
4. **Retrieval**: dada una pregunta, se recuperan los `k` fragmentos más similares
   por similitud coseno.


In [3]:
# --- 3.1 "Documentos" de origen: CV + proyectos (normalmente vendrían de PDFs) ---
RAW_DOCUMENTS = [
    {
        "id": "experiencia_data",
        "titulo": "Experiencia — Ingeniería de Datos",
        "texto": (
            "Aaron ha trabajado como ingeniero de datos diseñando pipelines ETL con "
            "Databricks, Spark y Delta Lake para procesar datos a escala. Ha construido "
            "arquitecturas medallion (bronze/silver/gold), optimizado jobs de Spark y "
            "orquestado flujos con Airflow y Databricks Workflows."
        ),
    },
    {
        "id": "experiencia_ai",
        "titulo": "Experiencia — Inteligencia Artificial y Agentes",
        "texto": (
            "Ha diseñado sistemas multi-agente con LLMs (GPT-4o-mini, modelos de Groq) "
            "aplicando patrones de orquestador y subagentes especializados, RAG sobre "
            "bases vectoriales, memoria conversacional y tool calling para automatizar "
            "tareas como cualificación de leads y análisis de datos."
        ),
    },
    {
        "id": "proyecto_portfolio_agents",
        "titulo": "Proyecto — Asistente de portfolio multi-agente",
        "texto": (
            "Este mismo proyecto: un chatbot con arquitectura multi-agente embebido en "
            "la web personal, con un subagente RAG para responder sobre la trayectoria "
            "profesional, un subagente de contacto que guarda leads en Airtable y un "
            "subagente analista que genera reportes a partir de CSV subidos por el "
            "visitante. Desplegado gratuitamente en Hugging Face Spaces con Docker."
        ),
    },
    {
        "id": "proyecto_databricks_bootcamp",
        "titulo": "Proyecto — Bootcamp de Databricks",
        "texto": (
            "Prácticas y ejercicios de bootcamp centrados en Databricks: notebooks, "
            "Delta Lake, control de acceso con Unity Catalog y buenas prácticas de "
            "ingeniería de datos en la nube."
        ),
    },
    {
        "id": "habilidades",
        "titulo": "Habilidades técnicas",
        "texto": (
            "Python, SQL, PySpark, Databricks, Delta Lake, Airflow, LLMs (OpenAI, Groq), "
            "arquitecturas RAG, bases de datos vectoriales (Supabase/pgvector), Docker, "
            "despliegue en Hugging Face Spaces y Streamlit Community Cloud."
        ),
    },
    {
        "id": "formacion",
        "titulo": "Formación",
        "texto": (
            "Formación continua en ingeniería de datos y en el ecosistema de agentes de "
            "IA: diseño de sistemas multi-agente, *tool calling*, *guardrails* y "
            "cumplimiento normativo (RGPD) en aplicaciones con datos de usuarios."
        ),
    },
]


def split_text(text: str, chunk_size: int = 40, overlap: int = 10) -> list[str]:
    """Trocea texto en fragmentos de `chunk_size` palabras con `overlap` de solape."""
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks


def build_chunks(documents: list[dict]) -> list[dict]:
    chunks = []
    for doc in documents:
        for i, chunk in enumerate(split_text(doc["texto"])):
            chunks.append({
                "doc_id": doc["id"],
                "titulo": doc["titulo"],
                "chunk_id": f"{doc['id']}_{i}",
                "texto": chunk,
            })
    return chunks


KNOWLEDGE_CHUNKS = build_chunks(RAW_DOCUMENTS)
print(f"{len(RAW_DOCUMENTS)} documentos -> {len(KNOWLEDGE_CHUNKS)} fragmentos indexables")


6 documentos -> 10 fragmentos indexables


In [4]:
# --- 3.2 Embeddings "hechos a mano" (bag-of-words + TF-IDF simplificado) ---
_TOKEN_RE = re.compile(r"[a-záéíóúñü0-9]+")


def tokenize(text: str) -> list[str]:
    return _TOKEN_RE.findall(text.lower())


def cosine_similarity(vec_a: Counter, vec_b: Counter) -> float:
    common = set(vec_a) & set(vec_b)
    dot = sum(vec_a[t] * vec_b[t] for t in common)
    norm_a = math.sqrt(sum(v * v for v in vec_a.values()))
    norm_b = math.sqrt(sum(v * v for v in vec_b.values()))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


class SimpleVectorStore:
    """
    Vector store minimalista en memoria pura Python.

    Sirve para demostrar el mecanismo de un RAG (embed -> almacenar -> buscar por
    similitud) sin depender de librerías externas. En producción este objeto se
    sustituye por una tabla de Supabase con la extensión `pgvector` y embeddings
    reales de OpenAI (`text-embedding-3-small`), manteniendo la misma interfaz
    `add_documents` / `search`.
    """

    def __init__(self):
        self._docs: list[dict] = []
        self._vectors: list[Counter] = []

    def add_documents(self, chunks: list[dict]) -> None:
        for chunk in chunks:
            self._docs.append(chunk)
            self._vectors.append(Counter(tokenize(chunk["texto"])))

    def search(self, query: str, k: int = 3) -> list[tuple[dict, float]]:
        query_vec = Counter(tokenize(query))
        scored = [
            (doc, cosine_similarity(query_vec, vec))
            for doc, vec in zip(self._docs, self._vectors)
        ]
        scored.sort(key=lambda pair: pair[1], reverse=True)
        return scored[:k]


vector_store = SimpleVectorStore()
vector_store.add_documents(KNOWLEDGE_CHUNKS)

# Prueba rápida de recuperación
for doc, score in vector_store.search("¿Qué experiencia tiene con Databricks y Spark?"):
    print(f"[{score:.2f}] {doc['titulo']} -> {doc['texto'][:80]}...")


[0.48] Experiencia — Ingeniería de Datos -> y orquestado flujos con Airflow y Databricks Workflows....
[0.45] Experiencia — Ingeniería de Datos -> Aaron ha trabajado como ingeniero de datos diseñando pipelines ETL con Databrick...
[0.23] Proyecto — Bootcamp de Databricks -> Prácticas y ejercicios de bootcamp centrados en Databricks: notebooks, Delta Lak...


<a id="4"></a>
## 4. Memoria de conversación (memoria de ventana)

Para mantener coherencia entre turnos sin disparar el consumo de tokens, se
conserva solo una **ventana deslizante** de los últimos N mensajes (5-10),
en lugar de todo el historial.


In [5]:
@dataclass
class ConversationMemory:
    """Memoria de ventana: conserva como máximo `window_size` turnos."""
    window_size: int = 8
    _messages: deque = field(default_factory=deque)

    def __post_init__(self):
        self._messages = deque(maxlen=self.window_size)

    def add(self, role: str, content: str) -> None:
        self._messages.append({"role": role, "content": content, "ts": datetime.now().isoformat()})

    def as_prompt_context(self) -> str:
        return "\n".join(f"{m['role']}: {m['content']}" for m in self._messages)

    def __len__(self):
        return len(self._messages)


memory = ConversationMemory(window_size=8)
memory.add("user", "Hola, quiero saber más sobre tu experiencia")
memory.add("assistant", "¡Claro! ¿Qué te gustaría saber en concreto?")
print(memory.as_prompt_context())


user: Hola, quiero saber más sobre tu experiencia
assistant: ¡Claro! ¿Qué te gustaría saber en concreto?


<a id="5"></a>
## 5. Herramientas (*Tool Calling*)

El orquestador y los subagentes pueden invocar funciones externas ("tools"). Aquí
se definen dos:

- `save_contact_to_airtable`: guarda un lead capturado en el chat (en la demo se
  escribe en un `.jsonl` local; en producción sería una llamada REST a la API de
  Airtable o Google Sheets, o un webhook de n8n).
- `analyze_dataframe`: recibe un `DataFrame` (CSV/Excel subido por el visitante) y
  genera estadísticas + un mini-informe de texto.


In [6]:
def save_contact_to_airtable(name: str, email: str, budget: str, interest: str) -> dict:
    """
    Guarda un contacto capturado por el subagente de Contacto.

    Demo: escribe una línea JSON en `demo_artifacts/contacts.jsonl`.

    Producción (comentado) — llamada real a la API REST de Airtable:

        import requests
        requests.post(
            f"https://api.airtable.com/v0/{BASE_ID}/{TABLE_NAME}",
            headers={"Authorization": f"Bearer {AIRTABLE_API_KEY}"},
            json={"fields": {"Nombre": name, "Email": email,
                              "Presupuesto": budget, "Interes": interest}},
        )

    También podría apuntar a un webhook de n8n que reenvíe el lead a Google Sheets,
    Slack o un CRM.
    """
    record = {
        "name": name, "email": email, "budget": budget,
        "interest": interest, "captured_at": datetime.now().isoformat(),
    }
    with open(DEMO_DIR / "contacts.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return record


def analyze_dataframe(rows: list[dict]) -> dict:
    """
    Analiza una lista de registros (equivalente a un CSV/Excel subido) sin
    depender de pandas, para que la demo funcione en cualquier entorno.
    Devuelve estadísticas básicas y una lista de observaciones ("insights").
    """
    if not rows:
        return {"error": "El archivo no contiene filas."}

    columns = list(rows[0].keys())
    numeric_cols = []
    stats = {}

    for col in columns:
        values = []
        for row in rows:
            try:
                values.append(float(row[col]))
            except (ValueError, TypeError):
                pass
        if len(values) == len(rows) and values:
            numeric_cols.append(col)
            stats[col] = {
                "min": min(values),
                "max": max(values),
                "media": sum(values) / len(values),
            }

    insights = [f"El archivo tiene {len(rows)} filas y {len(columns)} columnas."]
    for col, s in stats.items():
        insights.append(
            f"'{col}': media {s['media']:.2f} (mín {s['min']:.2f}, máx {s['max']:.2f})."
        )
    if not numeric_cols:
        insights.append("No se detectaron columnas numéricas para calcular estadísticas.")

    return {"n_filas": len(rows), "columnas": columns, "estadisticas": stats, "insights": insights}


<a id="6"></a>
## 6. Subagentes especializados

Cada subagente encapsula un objetivo concreto y expone un método `run(...)`.


In [7]:
RAG_SIMILARITY_THRESHOLD = 0.05  # por debajo de esto, el agente admite que no sabe


class RAGAgent:
    """Subagente experto en trayectoria: responde usando RAG sobre el CV/proyectos."""

    name = "Experto en Trayectoria (RAG)"

    def __init__(self, store: SimpleVectorStore):
        self.store = store

    def run(self, question: str) -> str:
        results = self.store.search(question, k=3)
        best_score = results[0][1] if results else 0.0

        # Guardrail: si no hay nada relevante, se admite explícitamente en vez de alucinar.
        if best_score < RAG_SIMILARITY_THRESHOLD:
            return (
                "No tengo información suficiente sobre eso en mi documentación. "
                "¿Quieres que te ponga en contacto directamente con Aaron?"
            )

        context = "\n".join(f"- {doc['texto']}" for doc, score in results if score > 0)
        system_prompt = (
            "Eres el asistente del portfolio de Aaron. Responde solo con la "
            "información de CONTEXTO proporcionada, en español, de forma breve y "
            "profesional. Si el contexto no basta, dilo con claridad."
        )
        user_prompt = f"CONTEXTO:\n{context}\n\nPREGUNTA: {question}"
        return call_llm(system_prompt, user_prompt)


class ContactAgent:
    """Subagente que captura datos de contacto y los envía a Airtable/Sheets."""

    name = "Gestor de Contacto"
    EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")

    def run(self, message: str) -> str:
        email_match = self.EMAIL_RE.search(message)
        if not email_match:
            return (
                "Para ponerte en contacto con Aaron necesito, al menos, tu email. "
                "¿Me lo compartes? (Al enviarlo aceptas que se use únicamente para "
                "responder a tu consulta, conforme al RGPD)."
            )

        email = email_match.group(0)
        name_match = re.search(r"(?:me llamo|soy)\s+([A-ZÁÉÍÓÚÑ][\w]+)", message, re.IGNORECASE)
        name = name_match.group(1) if name_match else "No indicado"
        budget_match = re.search(r"(\d+[\d.,]*\s?(?:€|eur|k))", message, re.IGNORECASE)
        budget = budget_match.group(1) if budget_match else "No indicado"

        record = save_contact_to_airtable(
            name=name, email=email, budget=budget, interest=message,
        )
        return (
            f"¡Gracias! He registrado tu contacto ({record['email']}) y Aaron te "
            "responderá en breve. Tus datos se guardan únicamente con este fin, "
            "conforme al RGPD, y puedes solicitar su borrado en cualquier momento."
        )


class DataAnalystAgent:
    """Subagente que analiza un CSV/Excel subido por el visitante."""

    name = "Analista de Datos"

    def run(self, rows: list[dict]) -> str:
        result = analyze_dataframe(rows)
        if "error" in result:
            return f"No he podido analizar el archivo: {result['error']}"
        report = "\n".join(f"- {line}" for line in result["insights"])
        return f"Informe automático del archivo subido:\n{report}"


<a id="7"></a>
## 7. Agente orquestador

Clasifica la intención del mensaje (por palabras clave en esta demo; en
producción esto lo haría el LLM mediante *function calling*/*tool choice*) y
enruta al subagente correspondiente. Registra qué subagente se invoca en cada
turno para dar **transparencia de proceso** en la interfaz.


In [8]:
INTENT_KEYWORDS = {
    "rag": ["experiencia", "proyecto", "cv", "trayectoria", "habilidad", "formación", "trabajo"],
    "contacto": ["contacto", "contactar", "presupuesto", "email", "correo", "colaborar", "contratar"],
    "datos": ["csv", "excel", "datos", "analiza", "analizar", "archivo", "informe"],
}


@dataclass
class OrchestratorAgent:
    """Agente principal: entiende la intención y delega en el subagente adecuado."""

    rag_agent: RAGAgent
    contact_agent: ContactAgent
    data_agent: DataAnalystAgent
    memory: ConversationMemory = field(default_factory=ConversationMemory)
    on_route: Callable[[str], None] | None = None  # hook para UI ("qué agente se invoca")

    def classify_intent(self, message: str) -> str:
        tokens = set(tokenize(message))
        scores = {
            intent: sum(1 for kw in kws if kw in tokens or kw in message.lower())
            for intent, kws in INTENT_KEYWORDS.items()
        }
        best_intent, best_score = max(scores.items(), key=lambda kv: kv[1])
        return best_intent if best_score > 0 else "desconocido"

    def handle_message(self, message: str, uploaded_rows: list[dict] | None = None) -> str:
        self.memory.add("user", message)
        try:
            intent = self.classify_intent(message)

            if intent == "datos" and uploaded_rows is not None:
                agent = self.data_agent
                reply = agent.run(uploaded_rows)
            elif intent == "contacto":
                agent = self.contact_agent
                reply = agent.run(message)
            elif intent == "rag":
                agent = self.rag_agent
                reply = agent.run(message)
            else:
                agent = None
                reply = (
                    "No estoy seguro de a qué subagente corresponde tu mensaje. "
                    "¿Quieres preguntar por mi trayectoria, dejar tus datos de "
                    "contacto o subir un archivo para analizarlo?"
                )

            if self.on_route:
                self.on_route(agent.name if agent else "Sin enrutar (fallback)")

        except Exception as exc:  # guardrail: nunca romper la conversación
            reply = "Ha ocurrido un error procesando tu mensaje. Inténtalo de nuevo, por favor."
            if self.on_route:
                self.on_route(f"Error: {exc}")

        self.memory.add("assistant", reply)
        return reply


<a id="8"></a>
## 8. Demo: conversación de ejemplo

Se instancia el orquestador con los tres subagentes y se simula una conversación
que recorre cada rama (RAG, contacto, análisis de datos y un caso "no lo sé").


In [9]:
def print_routing(agent_name: str) -> None:
    print(f"   🧭 Orquestador → subagente invocado: {agent_name}")


orchestrator = OrchestratorAgent(
    rag_agent=RAGAgent(vector_store),
    contact_agent=ContactAgent(),
    data_agent=DataAnalystAgent(),
    on_route=print_routing,
)

demo_turns = [
    "¿Qué experiencia tiene Aaron con Databricks y Spark?",
    "Me llamo Laura y mi email es laura@empresa.com, tengo un presupuesto de 5000€ para un proyecto de IA",
    "¿Cuál es la capital de Marte?",
]

for turn in demo_turns:
    print(f"👤 Usuario: {turn}")
    respuesta = orchestrator.handle_message(turn)
    print(f"🤖 Agente: {respuesta}\n")


👤 Usuario: ¿Qué experiencia tiene Aaron con Databricks y Spark?
   🧭 Orquestador → subagente invocado: Experto en Trayectoria (RAG)
🤖 Agente: [modo offline] He recibido tu consulta y la información recuperada por el subagente correspondiente. Con una API key real (OPENAI_API_KEY o GROQ_API_KEY) esta respuesta la redactaría el LLM en lenguaje natural.

👤 Usuario: Me llamo Laura y mi email es laura@empresa.com, tengo un presupuesto de 5000€ para un proyecto de IA
   🧭 Orquestador → subagente invocado: Gestor de Contacto
🤖 Agente: ¡Gracias! He registrado tu contacto (laura@empresa.com) y Aaron te responderá en breve. Tus datos se guardan únicamente con este fin, conforme al RGPD, y puedes solicitar su borrado en cualquier momento.

👤 Usuario: ¿Cuál es la capital de Marte?
   🧭 Orquestador → subagente invocado: Sin enrutar (fallback)
🤖 Agente: No estoy seguro de a qué subagente corresponde tu mensaje. ¿Quieres preguntar por mi trayectoria, dejar tus datos de contacto o subir un archivo par

<a id="9"></a>
## 9. Analista de datos: ejemplo con CSV

Simulamos que el visitante sube un CSV (aquí generado en memoria) con métricas de
un proyecto, y el subagente Analista de Datos genera un informe automático.


In [10]:
sample_csv_rows = [
    {"mes": "Enero", "usuarios_activos": 120, "coste_eur": 300, "conversion_pct": 2.1},
    {"mes": "Febrero", "usuarios_activos": 180, "coste_eur": 340, "conversion_pct": 2.4},
    {"mes": "Marzo", "usuarios_activos": 260, "coste_eur": 410, "conversion_pct": 3.0},
    {"mes": "Abril", "usuarios_activos": 300, "coste_eur": 430, "conversion_pct": 3.4},
]

print("👤 Usuario: Aquí tienes mis datos.csv, ¿puedes analizar el archivo?")
respuesta = orchestrator.handle_message(
    "¿Puedes analizar este archivo de datos?", uploaded_rows=sample_csv_rows
)
print(f"🤖 Agente: {respuesta}")


👤 Usuario: Aquí tienes mis datos.csv, ¿puedes analizar el archivo?
   🧭 Orquestador → subagente invocado: Analista de Datos
🤖 Agente: Informe automático del archivo subido:
- El archivo tiene 4 filas y 4 columnas.
- 'usuarios_activos': media 215.00 (mín 120.00, máx 300.00).
- 'coste_eur': media 370.00 (mín 300.00, máx 430.00).
- 'conversion_pct': media 2.73 (mín 2.10, máx 3.40).


<a id="10"></a>
## 10. Guardrails y cumplimiento (RGPD)

Dos mecanismos de seguridad ya están presentes arriba y se resumen aquí:

1. **Admitir el desconocimiento** (`RAGAgent.run`): si la similitud del mejor
   resultado recuperado está por debajo de un umbral, el agente no inventa una
   respuesta — ofrece derivar a contacto humano.
2. **Manejo de errores** (`OrchestratorAgent.handle_message`): cualquier excepción
   se captura y se traduce en un mensaje controlado, sin romper la conversación
   ni exponer trazas internas al usuario.

Además, el subagente de Contacto:

- Informa explícitamente de que el visitante está interactuando con una IA.
- Solo solicita los datos mínimos necesarios (email, y opcionalmente nombre y
  presupuesto).
- Indica la base y finalidad del tratamiento de datos (RGPD) y el derecho a
  solicitar su borrado.


In [11]:
def gdpr_notice() -> str:
    return (
        "🔒 Aviso: estás hablando con un asistente de IA, no con una persona. "
        "Si compartes tus datos de contacto, se almacenarán únicamente para "
        "responder a tu consulta, conforme al RGPD, y puedes solicitar su "
        "eliminación en cualquier momento escribiendo a Aaron."
    )


print(gdpr_notice())


🔒 Aviso: estás hablando con un asistente de IA, no con una persona. Si compartes tus datos de contacto, se almacenarán únicamente para responder a tu consulta, conforme al RGPD, y puedes solicitar su eliminación en cualquier momento escribiendo a Aaron.


<a id="11"></a>
## 11. Despliegue gratuito

**Opción recomendada — Hugging Face Spaces + Docker:**

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 7860
CMD ["python", "app.py"]
```

```
requirements.txt
-----------------
fastapi
uvicorn
openai
groq
```

**Alternativa 100% Python — Streamlit Community Cloud:**

```bash
pip install streamlit
streamlit run app.py
```

Ambas opciones tienen *tier* gratuito y permiten exponer un endpoint HTTP (o una
app web completa) que el `demo.html` de este proyecto puede consumir vía
`fetch()`.


<a id="12"></a>
## 12. Integración en la web (embed)

Una vez desplegado el backend, se genera un snippet embebible y se coloca justo
antes de `</body>` en el HTML de la web personal:

```html
<script>
  window.PORTFOLIO_AGENT_CONFIG = {
    endpoint: "https://tu-espacio.hf.space/chat",
    primaryColor: "#2a78d6",
    position: "bottom-right"
  };
</script>
<script src="https://tu-cdn/portfolio-agent-widget.js"></script>
```

Recomendaciones de UX aplicadas en `demo.html` (adjunto en este mismo proyecto):

- **Burbuja flotante** en la esquina inferior derecha (patrón estándar de chat).
- **CSS personalizado** — colores y tipografía alineados con la marca personal,
  sin usar el estilo por defecto de ningún framework.
- **Transparencia de proceso** — se muestra qué subagente está respondiendo en
  cada turno ("🔎 Subagente: Experto en Trayectoria").
- **Aviso de IA + RGPD** visible antes de capturar cualquier dato personal.
- **Enlace a la documentación técnica** (este mismo repositorio de GitHub), para
  que cualquier reclutador pueda validar la arquitectura.


<a id="13"></a>
## 13. Conclusión

Este notebook demuestra, de punta a punta y sin dependencias pesadas, las piezas
clave de un sistema multi-agente de nivel productivo:

- Arquitectura de **orquestador + subagentes** especializados.
- **RAG** completo (chunking, embeddings, vector store, retrieval, guardrail de
  "no lo sé").
- **Memoria de ventana** para conversaciones coherentes y económicas en tokens.
- **Tool calling** hacia sistemas externos (Airtable/Sheets) y hacia análisis de
  datos (CSV/Excel).
- **Guardrails** (manejo de errores, umbral de confianza) y **cumplimiento RGPD**.
- Ruta de **despliegue gratuito** (Hugging Face Spaces / Streamlit) e
  **integración embebida** en una web estática.

El archivo `demo.html` que acompaña a este notebook es una demostración
autocontenida (sin backend) del mismo patrón de interacción, pensada para
incrustarse directamente en una página de portfolio.
